# Week 2 Exercise — Technical Tutor (full prototype)

Upgrade of the Week 1 technical Q&A: **Gradio UI**, **streaming**, **system prompt** for expertise, **model switching** (OpenRouter + optional Ollama), and one **tool** (current time) so the tutor can reference "what time it is" when explaining.

Uses **OpenRouter** for cloud models. Set `OPENROUTER_API_KEY` in `.env`. For Ollama, run `ollama pull llama3.2` and have Ollama running.

In [ ]:
import os
import json
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    print("Add OPENROUTER_API_KEY to .env")
else:
    print("OpenRouter API key loaded.")

# Model options: OpenRouter IDs + optional Ollama
MODELS = {
    "OpenRouter: gpt-4o-mini": ("openai/gpt-4o-mini", "https://openrouter.ai/api/v1", api_key),
    "OpenRouter: claude-3-haiku": ("anthropic/claude-3-haiku", "https://openrouter.ai/api/v1", api_key),
    "Ollama: llama3.2": ("llama3.2", "http://localhost:11434/v1", "ollama"),
}

def get_client(model_key):
    model_id, base_url, key = MODELS.get(model_key, list(MODELS.values())[0])
    return OpenAI(api_key=key or "ollama", base_url=base_url), model_id
# (MODELS values are tuples: model_id, base_url, api_key)

In [ ]:
SYSTEM_PROMPT = """You are a helpful technical tutor. You answer questions about Python, software engineering, data science, and LLMs.
You explain clearly and use examples when useful. If the user asks what time it is or for the current time, use the get_current_time tool.
Respond in markdown. Be concise but thorough."""

get_current_time_tool = {
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": "Get the current date and time (e.g. when the user asks what time it is).",
        "parameters": {
            "type": "object",
            "properties": {},
            "additionalProperties": False
        }
    }
}

def get_current_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def handle_tool_calls(message):
    responses = []
    for tc in message.tool_calls:
        if tc.function.name == "get_current_time":
            responses.append({"role": "tool", "content": get_current_time(), "tool_call_id": tc.id})
    return responses

In [ ]:
def chat_stream(message, history, model_key):
    client, model_id = get_client(model_key)
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]
    tools = [get_current_time_tool]

    response = client.chat.completions.create(model=model_id, messages=messages, tools=tools, stream=True)
    full_content = ""
    for chunk in response:
        delta = chunk.choices[0].delta if chunk.choices else None
        if delta and getattr(delta, "content", None):
            full_content += delta.content or ""
            yield full_content
        if delta and getattr(delta, "tool_calls", None) and delta.tool_calls:
            break

    # If we stopped for tool_calls, we need to do a non-streaming round (simplified: one round)
    if not full_content.strip():
        messages_final = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]
        resp = client.chat.completions.create(model=model_id, messages=messages_final, tools=tools)
        if resp.choices[0].finish_reason == "tool_calls":
            msg = resp.choices[0].message
            messages_final.append(msg)
            messages_final.extend(handle_tool_calls(msg))
            resp2 = client.chat.completions.create(model=model_id, messages=messages_final, tools=tools)
            yield resp2.choices[0].message.content
        else:
            yield resp.choices[0].message.content

In [ ]:
def chat(message, history, model_key):
    client, model_id = get_client(model_key)
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]
    tools = [get_current_time_tool]

    response = client.chat.completions.create(model=model_id, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        messages.append(msg)
        messages.extend(handle_tool_calls(msg))
        response = client.chat.completions.create(model=model_id, messages=messages, tools=tools)

    return response.choices[0].message.content

In [ ]:
model_dropdown = gr.Dropdown(
    choices=list(MODELS.keys()),
    value=list(MODELS.keys())[0],
    label="Model"
)

gr.ChatInterface(
    fn=chat,
    additional_inputs=[model_dropdown],
    type="messages",
    title="Technical Tutor (Week 2)",
    description="Ask a technical question. Switch models and try: "What time is it?" to use the tool.",
    examples=[
        ["Explain what a Python generator is and give a short example.", list(MODELS.keys())[0]],
        ["What time is it?", list(MODELS.keys())[0]],
    ],
).launch()